In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/customer_churn_dataset-testing-master.csv')

In [ ]:
df.shape

(64374, 12)

In [ ]:
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [ ]:
df.isna().sum()

,0
CustomerID,0
Age,0
Gender,0
Tenure,0
Usage Frequency,0
Support Calls,0
Payment Delay,0
Subscription Type,0
Contract Length,0
Total Spend,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64374 entries, 0 to 64373
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   CustomerID         64374 non-null  int64 
 1   Age                64374 non-null  int64 
 2   Gender             64374 non-null  object
 3   Tenure             64374 non-null  int64 
 4   Usage Frequency    64374 non-null  int64 
 5   Support Calls      64374 non-null  int64 
 6   Payment Delay      64374 non-null  int64 
 7   Subscription Type  64374 non-null  object
 8   Contract Length    64374 non-null  object
 9   Total Spend        64374 non-null  int64 
 10  Last Interaction   64374 non-null  int64 
 11  Churn              64374 non-null  int64 
dtypes: int64(9), object(3)
memory usage: 5.9+ MB


In [ ]:
# Install ydata-profiling if not already installed
# !pip install ydata-profiling

import ydata_profiling as pp

report = pp.ProfileReport(df)

#saving the report
report.to_file(output_file='report.html')

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ','_')

In [ ]:
df.drop(columns=['customerid'],inplace=True)

In [ ]:
df.sample(5)

,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn
41740,63,Male,20,16,2,10,Standard,Monthly,200,4,0
57316,18,Male,59,20,9,29,Basic,Monthly,695,23,1
18301,23,Male,1,14,5,7,Premium,Quarterly,950,26,0
6986,44,Female,35,20,7,26,Standard,Monthly,138,17,1
2429,46,Female,59,14,6,12,Standard,Quarterly,153,17,0


In [ ]:
cat_cols = ['gender','subscription_type','contract_length']
num_cols = [col for col in df.columns if col not in cat_cols]

In [ ]:
cat_cols , num_cols

(['gender', 'subscription_type', 'contract_length'],
 ['age',
  'tenure',
  'usage_frequency',
  'support_calls',
  'payment_delay',
  'total_spend',
  'last_interaction',
  'churn'])

In [ ]:
def get_categories(cat_cols):
  for col in cat_cols:
     print(f'{col} :',df[col].value_counts())

get_categories(cat_cols)

gender : gender
Female    34353
Male      30021
Name: count, dtype: int64
subscription_type : subscription_type
Standard    21502
Basic       21451
Premium     21421
Name: count, dtype: int64
contract_length : contract_length
Monthly      22130
Annual       21410
Quarterly    20834
Name: count, dtype: int64


In [ ]:
def get_distribution_of_churn(df):
   print(df['churn'].value_counts())

get_distribution_of_churn(df)

churn
0    33881
1    30493
Name: count, dtype: int64


In [ ]:
def get_outliers(num_cols):
  for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].value_counts()

    print(f'Outliers in {col}:')
    if outliers.empty:
      print('no outliers detected')
    else:
      print(outliers)
    print('\n')

get_outliers(num_cols)

Outliers in age:
no outliers detected


Outliers in tenure:
no outliers detected


Outliers in usage_frequency:
no outliers detected


Outliers in support_calls:
no outliers detected


Outliers in payment_delay:
no outliers detected


Outliers in total_spend:
no outliers detected


Outliers in last_interaction:
no outliers detected


Outliers in churn:
no outliers detected




In [ ]:
# maps for cat_columns

maps = {
    'gender': {'Male': 0, 'Female': 1},
    'subscription_type': {'Basic': 0, 'Premium': 1, 'Standard': 2},
    'contract_length': {'Monthly': 0, 'Annual': 1, 'Quarterly': 2}
}

def get_map_cat_cols(cat_cols):
  for col in cat_cols:
    df[col] = df[col].map(maps[col])

get_map_cat_cols(cat_cols)

In [ ]:
#splitting X and y
X = df.drop(columns=['churn'])
y = df['churn']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

model = LogisticRegression()
model.fit(X_train,y_train)

print(accuracy_score(y_test,model.predict(X_test)))
print(classification_report(y_test,model.predict(X_test)))

0.8319223300970874
              precision    recall  f1-score   support

           0       0.85      0.83      0.84      6793
           1       0.82      0.83      0.82      6082

    accuracy                           0.83     12875
   macro avg       0.83      0.83      0.83     12875
weighted avg       0.83      0.83      0.83     12875



In [ ]:
# random forest
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train,y_train)

print(accuracy_score(y_test,model.predict(X_test)))
print(classification_report(y_test,model.predict(X_test)))

0.9992233009708738
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6793
           1       1.00      1.00      1.00      6082

    accuracy                           1.00     12875
   macro avg       1.00      1.00      1.00     12875
weighted avg       1.00      1.00      1.00     12875



In [74]:
import pickle
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('model',RandomForestClassifier())
])

pipe.fit(X_train,y_train)

pickle.dump(pipe,open('model.pkl','wb'))

In [ ]:
from xgboost import XGBClassifier
model = XGBClassifier()
model.fit(X_train,y_train)

print(accuracy_score(y_test,model.predict(X_test)))

0.9998446601941747


In [ ]:
import lightgbm as lgb
model = lgb.LGBMClassifier()
model.fit(X_train,y_train)

print(accuracy_score(y_test,model.predict(X_test)))
print(classification_report(y_test,model.predict(X_test)))

[LightGBM] [Info] Number of positive: 24411, number of negative: 27088
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003459 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 482
[LightGBM] [Info] Number of data points in the train set: 51499, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.474009 -> initscore=-0.104057
[LightGBM] [Info] Start training from score -0.104057


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.9999223300970874
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6793
           1       1.00      1.00      1.00      6082

    accuracy                           1.00     12875
   macro avg       1.00      1.00      1.00     12875
weighted avg       1.00      1.00      1.00     12875

